# 0015 — SQL Publish rereview views

Spark View는 SQL Analytics Endpoint에 노출되지 않으므로, Web Sync가 조회할 T-SQL View를 생성한다.

In [ ]:
sql_endpoint_server = "6bllex7s7pjebaovxla3glcbtu-x32j73qbjlgubmmuvexgtmaxdi.datawarehouse.fabric.microsoft.com"
sql_endpoint_database = "watchback_lakehouse_dev_v2"
applied_by_run_id = None


In [ ]:
import hashlib
import json
import struct
from datetime import datetime, timezone

import pyodbc
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, TimestampType

MIGRATION_VERSION = 15
MIGRATION_NAME = "create-sql-publish-rereview-views"
MIGRATION_TABLE = "gold._schema_migrations"
EXPECTED_PRIOR_VERSION = 14
EXPECTED_PRIOR_NAME = "create-ds-gold-rereview-tables"
TOKEN_SCOPE = "https://database.windows.net/"
SQL_COPT_SS_ACCESS_TOKEN = 1256

VIEW_DDLS = {
    "gold.publish_rereview_batch": """
CREATE OR ALTER VIEW gold.publish_rereview_batch AS
SELECT batch_id, ds_run_id, status, as_of, generated_at, policy_version, schema_version, candidate_count
FROM gold.gold_rereview_batch
WHERE status = 'completed'
""",
    "gold.publish_rereview_candidate": """
CREATE OR ALTER VIEW gold.publish_rereview_candidate AS
SELECT candidate.batch_id, candidate.candidate_id,
       CASE WHEN candidate.cve_id IS NOT NULL THEN 'CVE' ELSE 'GHSA' END AS subject_type,
       COALESCE(candidate.cve_id, candidate.ghsa_id) AS subject_id,
       candidate.ghsa_id, candidate.cve_id, candidate.priority,
       candidate.primary_trigger_type AS change_type,
       candidate.reason_summary, candidate.selected_at
FROM gold.gold_rereview_candidate AS candidate
JOIN gold.gold_rereview_batch AS batch ON batch.batch_id = candidate.batch_id
WHERE batch.status = 'completed'
""",
    "gold.publish_rereview_candidate_evidence": """
CREATE OR ALTER VIEW gold.publish_rereview_candidate_evidence AS
SELECT evidence.batch_id, evidence.evidence_id, evidence.candidate_id,
       evidence.signal, evidence.value_type, evidence.previous_value, evidence.current_value,
       evidence.evidence_observed_at AS observed_at, evidence.source, evidence.evidence_summary
FROM gold.gold_rereview_candidate_evidence AS evidence
JOIN gold.gold_rereview_batch AS batch ON batch.batch_id = evidence.batch_id
WHERE batch.status = 'completed'
""",
}

EXPECTED_COLUMNS = {
    "publish_rereview_batch": ["batch_id", "ds_run_id", "status", "as_of", "generated_at", "policy_version", "schema_version", "candidate_count"],
    "publish_rereview_candidate": ["batch_id", "candidate_id", "subject_type", "subject_id", "ghsa_id", "cve_id", "priority", "change_type", "reason_summary", "selected_at"],
    "publish_rereview_candidate_evidence": ["batch_id", "evidence_id", "candidate_id", "signal", "value_type", "previous_value", "current_value", "observed_at", "source", "evidence_summary"],
}
SPEC = {"view_ddls": VIEW_DDLS, "expected_columns": EXPECTED_COLUMNS}
CHECKSUM = hashlib.sha256(json.dumps(SPEC, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

if not sql_endpoint_server:
    raise ValueError("sql_endpoint_server parameter is required")
if not spark.catalog.tableExists(MIGRATION_TABLE):
    raise RuntimeError("migration history missing")

history_rows = spark.table(MIGRATION_TABLE).select(F.col("version").cast("int").alias("version"), "name", "checksum").collect()
history = {int(row["version"]): row for row in history_rows}
if len(history) != len(history_rows):
    raise RuntimeError("duplicate migration history version")
if EXPECTED_PRIOR_VERSION not in history or history[EXPECTED_PRIOR_VERSION]["name"] != EXPECTED_PRIOR_NAME:
    raise RuntimeError(f"{EXPECTED_PRIOR_VERSION:04d} {EXPECTED_PRIOR_NAME} must be applied first")
if any(version > MIGRATION_VERSION for version in history):
    raise RuntimeError(f"database is ahead of migration {MIGRATION_VERSION:04d}")
already_applied = MIGRATION_VERSION in history
if already_applied and (history[MIGRATION_VERSION]["name"] != MIGRATION_NAME or history[MIGRATION_VERSION]["checksum"] != CHECKSUM):
    raise RuntimeError(f"applied migration {MIGRATION_VERSION:04d} differs from definition")

token = notebookutils.credentials.getToken(TOKEN_SCOPE).encode("utf-16-le")
token_struct = struct.pack(f"<I{len(token)}s", len(token), token)
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER=tcp:{sql_endpoint_server},1433;DATABASE={sql_endpoint_database};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
)
connection = pyodbc.connect(connection_string, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct}, autocommit=True)
try:
    cursor = connection.cursor()
    for ddl in VIEW_DDLS.values():
        cursor.execute(ddl)

    for view_name, expected in EXPECTED_COLUMNS.items():
        cursor.execute("""
            SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = 'gold' AND TABLE_NAME = ?
            ORDER BY ORDINAL_POSITION
        """, (view_name,))
        actual = [row[0] for row in cursor.fetchall()]
        if actual != expected:
            raise RuntimeError(f"SQL view column mismatch: {view_name}: actual={actual}, expected={expected}")

    cursor.execute("SELECT COUNT(*) FROM gold.publish_rereview_batch WHERE status <> 'completed' OR status IS NULL")
    if cursor.fetchone()[0] != 0:
        raise RuntimeError("publish batch contains non-completed rows")
    cursor.execute("SELECT COUNT(*) FROM gold.publish_rereview_candidate c LEFT JOIN gold.publish_rereview_batch b ON b.batch_id=c.batch_id WHERE b.batch_id IS NULL")
    if cursor.fetchone()[0] != 0:
        raise RuntimeError("publish candidate has missing batch")
    cursor.execute("SELECT COUNT(*) FROM gold.publish_rereview_candidate_evidence e LEFT JOIN gold.publish_rereview_candidate c ON c.batch_id=e.batch_id AND c.candidate_id=e.candidate_id WHERE c.candidate_id IS NULL")
    if cursor.fetchone()[0] != 0:
        raise RuntimeError("publish evidence has missing candidate")

    counts = {}
    for key, view_name in (("batches", "publish_rereview_batch"), ("candidates", "publish_rereview_candidate"), ("evidence", "publish_rereview_candidate_evidence")):
        cursor.execute(f"SELECT COUNT(*) FROM gold.{view_name}")
        counts[key] = cursor.fetchone()[0]
finally:
    connection.close()

if not already_applied:
    migration_schema = StructType([
        StructField("version", IntegerType(), False),
        StructField("name", StringType(), False),
        StructField("applied_at", TimestampType(), False),
        StructField("applied_by_run_id", StringType(), True),
        StructField("checksum", StringType(), False),
    ])
    row = [(MIGRATION_VERSION, MIGRATION_NAME, datetime.now(timezone.utc).replace(tzinfo=None), applied_by_run_id, CHECKSUM)]
    spark.createDataFrame(row, migration_schema).write.mode("append").saveAsTable(MIGRATION_TABLE)

run_result = {
    "status": "skipped" if already_applied else "ok",
    "migration_version": MIGRATION_VERSION,
    "migration_name": MIGRATION_NAME,
    "checksum": CHECKSUM,
    "views": list(VIEW_DDLS),
    "counts": counts,
}
print(json.dumps(run_result, sort_keys=True))
notebookutils.notebook.exit(json.dumps(run_result, sort_keys=True))
